In [5]:
import sys
sys.path.append('..')

print("Đang load thư viện PyTorch (Sẽ hơi lâu một chút)...")
import torch
import torch.nn as nn
import torchvision
from src.utils import get_device, extract_features
print("Đã load xong PyTorch!")

Đang load thư viện PyTorch (Sẽ hơi lâu một chút)...
Đã load xong PyTorch!


In [6]:
device = get_device()
print(f"[INFO] Đang chạy trên thiết bị: {device}")

print("Đang tải mạng ResNet chuyên dụng cho CIFAR-10...")
# Tải mô hình đã được train sẵn cho CIFAR-10
resnet = torch.hub.load("chenyaofo/pytorch-cifar-models", "cifar10_resnet20", pretrained=True)

# Tách bỏ lớp phân loại cuối cùng để lấy vector đặc trưng
backbone = nn.Sequential(*list(resnet.children())[:-1]).to(device)

print("Đang chuẩn bị Transform và tải dữ liệu CIFAR-10...")
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# Ảnh CIFAR-10 gốc là 32x32, mạng này thiết kế chuẩn cho 32x32 nên KHÔNG CẦN Resize lên 224 nữa
transform_cifar = transforms.Compose([
    transforms.ToTensor(),         
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)) # Chuẩn hóa riêng của CIFAR-10
])

cifar_data = torchvision.datasets.CIFAR10(root='../data', train=False, download=True, transform=transform_cifar)
cifar_subset = torch.utils.data.Subset(cifar_data, range(5000))
cifar_loader = DataLoader(cifar_subset, batch_size=128, shuffle=False)

print("[INFO] Đang trích xuất đặc trưng (Sẽ siêu nhanh vì ảnh nhỏ)...")
z_cifar, y_cifar = extract_features(cifar_loader, backbone, device)
print("Trích xuất xong!")

[INFO] Đang chạy trên thiết bị: cpu
Đang tải mạng ResNet chuyên dụng cho CIFAR-10...


Using cache found in C:\Users\Admin/.cache\torch\hub\chenyaofo_pytorch-cifar-models_master


Đang chuẩn bị Transform và tải dữ liệu CIFAR-10...
[INFO] Đang trích xuất đặc trưng (Sẽ siêu nhanh vì ảnh nhỏ)...
Trích xuất xong!


In [ ]:
from src.utils import train_clustering

print("\n" + "="*50)
print("--- THỰC NGHIỆM 1: TÁI HIỆN CIFAR-10 (BẢN TỐI ƯU) ---")
print("[INFO] Đang tiến hành gom cụm (Clustering). Vui lòng đợi vài giây...")

# Chạy thuật toán PnP với số vòng lặp lớn hơn (150) và tốc độ học nhỏ (0.01)
k_star, acc, nmi, ari = train_clustering(
    features=z_cifar, 
    labels=y_cifar, 
    k_max=30, 
    device=device, 
    apply_sparsity=True,
    epochs=150,    
    lr=0.01          
)

print("\n[HOÀN THÀNH]")
print(f"-> K_max khởi tạo: 30 | K* tự động tìm được: {k_star}")
print(f"-> KẾT QUẢ: ACC = {acc*100:.2f}% | NMI = {nmi*100:.2f}% | ARI = {ari*100:.2f}%")
print("="*50)


--- THỰC NGHIỆM 1: TÁI HIỆN CIFAR-10 (BẢN TỐI ƯU) ---
[INFO] Đang tiến hành gom cụm (Clustering). Vui lòng đợi vài giây...

[HOÀN THÀNH]
-> K_max khởi tạo: 30 | K* tự động tìm được: 6
-> KẾT QUẢ: ACC = 56.70% | NMI = 68.07% | ARI = 43.62%
